
  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%">


# Summer Institute in Computational Social Sciences - Buenos Aires 2026
# Taller: Procesamiento de Lenguaje Natural y polarización
# Anotación y LLMs

### Profesor: Juan Manuel Pérez




En esta notebook vamos a jugar un poco con la API de Gemini.

Regístrense en [la página de Gemini/Google AI Studio y generen una API KEY](https://ai.google.dev/gemini-api/docs/api-key)

Una vez que la generen, la ponen acá (no la copypasteo por obvias razones)

In [ ]:
!pip install google-genai datasets -qqq

In [ ]:
import getpass
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
client = genai.Client(api_key=userdata.get("gemini-test"))

## Zero-shot classification

Podemos usar un LLM para que haga tareas para las cuales no fue entrenado. Por ejemplo, podemos usarlo para clasificar textos.

Para eso, usaremos un prompt que le diga qué hacer. Por ejemplo, si le decimos "Clasificá este texto como positivo o negativo", y le damos un texto, nos va a decir si es positivo o negativo.

In [ ]:

text = """Clasificar el siguiente texto en positivo, negativo, o neutral. Pensá paso a paso la respuesta y contestá 'La respuesta final es' y el sentimiento (uno de los siguientes: positivo, negativo, neutral) al final de la respuesta. No respondas nada más que eso.

texto: MESSI SOS EL GOAT LPM"""

model_name = "gemini-3.5-flash"
response = client.models.generate_content(
    model=model_name, contents=text
)
print(response.text)


Para clasificar el texto "MESSI SOS EL GOAT LPM", realizamos el siguiente análisis paso a paso:

1. **Identificación de términos clave:** El texto menciona a "MESSI" y utiliza la sigla "GOAT" (*Greatest Of All Time* o "El mejor de todos los tiempos"), que es un término de máximo elogio y admiración en el ámbito deportivo.
2. **Análisis de la expresión "SOS EL GOAT":** Es una afirmación directa y muy elogiosa que califica al sujeto de manera extremadamente favorable.
3. **Análisis de la sigla "LPM" (La Puta Madre):** Aunque literalmente es una expresión vulgar, en el contexto informal y futbolístico rioplatense se utiliza aquí como un intensificador de euforia, emoción y admiración, no como un insulto o queja. Expresa un entusiasmo desbordante.
4. **Conclusión del sentimiento:** Dado que todo el mensaje expresa una profunda admiración, alegría y apoyo entusiasta hacia el deportista, el sentimiento general es claramente favorable.

La respuesta final es positivo


## Json mode

In [ ]:
from typing import Literal
from enum import Enum
from pydantic import BaseModel, Field

class HateCategories(str, Enum):
    WOMEN = "WOMEN"
    LGBTI = "LGBTI"
    RACISM = "RACISM"
    CLASS = "CLASS"
    POLITICS = "POLITICS"
    DISABLED = "DISABLED"
    APPEARANCE = "APPEARANCE"
    CRIMINAL = "CRIMINAL"

class CommentLabel(BaseModel):
    HATEFUL: bool = Field(
        description="El comentario contiene discurso de odio explícito hacia un grupo."
    )
    OFFENSIVE: bool = Field(
        description=(
            "El comentario contiene lenguaje ofensivo, insultos o agresividad, "
            "independientemente de si es discurso de odio."
        )
    )
    CALLS: bool = Field(
        description="Incita a actuar en contra de alguien. Solo aplica si HATEFUL=True."
    )
    categories: list[HateCategories]


TEMPLATE_PROMPT = """Tu tarea es detectar discurso de odio en comentarios de Twitter a noticias periodísticas.

Definiciones:
- HATEFUL: discurso de odio explícito hacia un grupo (por género, raza, orientación sexual, etc.)
- OFFENSIVE: lenguaje ofensivo, insultos o agresividad, independientemente de si es discurso de odio.
- CALLS: incitación a actuar en contra de alguien (solo si HATEFUL=True)
- Las categorías WOMEN/LGBTI/RACISM/CLASS/POLITICS/DISABLED/APPEARANCE/CRIMINAL
  solo aplican si HATEFUL=True; marcalas en False si HATEFUL=False.

Definiciones de las categorías:
- WOMEN: Sexismo o misoginia
- LGBTI: Homofobia o transfobia
- RACISM: Racismo o xenofobia
- CLASS: Discriminación por clase social
- POLITICS: Odio por afiliación política
- DISABLED: Discriminación por discapacidad
- APPEARANCE: Ataque a la apariencia física
- CRIMINAL: discurso de odio contra personas privadas de libertad

Responder con un JSON con los siguientes campos:
{{
    "HATEFUL": bool,
    "OFFENSIVE": bool,
    "CALLS": bool,
    "categories": list[str]
}}

---

Contexto: {contexto}
Comentario: {comentario}
"""
contexto = '''Repudian una serie de declaraciones de Eduardo Feinmann sobre los chinos: "El planeta se va a ocupar de ustedes"'''
comentario = '''@usuario Todos los políticamente correctos se dan cuenta que hay una pandemia por culpa de los chinos?hay que hacer algo con esta gente urgente...'''

prompt = TEMPLATE_PROMPT.format(contexto=contexto, comentario=comentario)
response = client.models.generate_content(
    model=model_name, contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=CommentLabel,
    ),
)

# Probar agregar CoT

print(response.text)

{"HATEFUL": true, "OFFENSIVE": true, "CALLS": true, "categories": ["RACISM"]}


In [ ]:
response.parsed

CommentLabel(HATEFUL=True, OFFENSIVE=True, CALLS=True, categories=[<HateCategories.RACISM: 'RACISM'>])

In [ ]:
response = client.models.generate_content(
  model="gemini-3.5-flash",
  contents=prompt,
  config=types.GenerateContentConfig(
    response_mime_type='application/json',
    response_schema=CommentLabel,
    thinking_config=types.ThinkingConfig(
      include_thoughts=True
    )
  )
)

In [ ]:
response.candidates[0].content.parts

[Part(
   text="""{
   "HATEFUL": false,
   "OFFENSIVE": true,
   "CALLS": false,
   "categories": []
 }""",
   thought_signature=b'\x12^\n\\\x01\x11M2\x0f\x05\x95\xfe\xaa\x9d\xf3~\xea\xf6\x8aO5J\xff\xf6\xbc\xfa\xdf\xa4\xd9\xfe\xa65;\x85a8\x8a\xe2\xd4"$\xa9\x14+]q>@wX\x05\xef\x14l\xfe7\xc0\xf8\xb6\x98s\xc6 K\xdb\x81\xd88x\xbdV\x95\x8aC\xff\x8f7\xbf-\xc9K\x98\x7f\xf3\x18,$\xdf\x96\rum\xc0\x91o\x92'
 )]

In [ ]:
!curl https://raw.githubusercontent.com/finiteautomata/sicss/refs/heads/main/data/sample.json -o sample.json # nos bajamos los datitos

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  470k  100  470k    0     0  3549k      0 --:--:-- --:--:-- --:--:-- 3536k


In [ ]:
import json

data = []
for line in open("sample.json"):
    data.append(json.loads(line.strip()))

In [ ]:
from tqdm import tqdm
import time

for example in tqdm(data):
    if "response" in example:
        continue
    contexto = example["context_tweet"]
    text = example["text"]
    prompt = TEMPLATE_PROMPT.format(contexto=contexto, comentario=text)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=CommentLabel,
            thinking_config=types.ThinkingConfig(
            include_thoughts=True
            )
        )
    )

    example["response"] = response

    time.sleep(5)


100%|██████████| 1/1 [00:05<00:00,  5.64s/it]
